# Legal corpus vectorization on Kaggle

Embeds the JSONL that `scripts/legal_data/fetch_*.py` produced (laws, regulations, Supreme Court rulings) with
bge-m3 into Chroma: one store per category, separate from the Legal tab's signed index. Code:
`scripts/legal_data/vectorize.py`, `src/docslides/legal_data/`.

**Before running** (panel on the right):
1. On your machine: `python scripts/legal_data/package_for_kaggle.py` (optionally `--categories ...`), then upload
   `legal_txt/_kaggle_upload/legal_corpus_*.zip` as a **private** Kaggle Dataset and attach it (*Add Input*).
2. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **On** (repo, packages, the bge-m3 weights).
3. *Add-ons -> Secrets*: tick **GITHUB_TOKEN** (can read `zananiri/AI-IZ`).
4. Set the cell below. Kaggle keeps at most 20 GB of output and runs 12 h per session, so do one category group
   per session: `laws,procedural_rules` first, then `supreme_court` (split with `SHARD` if the dry run says it's
   too big; merge the shards afterwards with `vectorize.py --merge-from`).
5. *Save Version -> Save & Run All*. Download `legal_corpus_<category>*.zip` from the version's *Output*.

In [ ]:
BRANCH = "main"
CATEGORIES = "laws,procedural_rules"   # a later session: "supreme_court"
SHARD = None                           # e.g. "0/2", then "1/2" in the next session
SAMPLE_FIRST = True                    # index 50 records into a scratch store and probe them first
DEVICES = "cuda:0,cuda:1"              # both T4s; "cuda" for one
RESUME_FROM = None                     # a previous session's output zip (attached as input) to continue from

## 1. Code and packages

In [ ]:
import os, pathlib, shutil, subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/AI-IZ"
!git clone -q --depth 1 -b {BRANCH} https://{token}@github.com/zananiri/AI-IZ.git {REPO}
!git -C {REPO} remote set-url origin https://github.com/zananiri/AI-IZ.git
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[legal,legal-data,dev]"

## 2. Input and output locations

In [ ]:
roots = [p for p in pathlib.Path("/kaggle/input").rglob("*")
         if p.is_dir() and any((p / c).is_dir() for c in ("laws", "procedural_rules", "supreme_court"))]
assert roots, "attach the uploaded corpus dataset (see the first cell)"
INPUT = str(roots[0])
print("input:", INPUT, sorted(os.listdir(INPUT)))
# Built in /tmp (scratch space); only the zips go to /kaggle/working (the 20 GB saved output).
VECTORDB = "/tmp/legal_corpus_vectordb"
pathlib.Path(VECTORDB).mkdir(parents=True, exist_ok=True)
if RESUME_FROM:
    shutil.unpack_archive(RESUME_FROM, VECTORDB)  # the zip holds <category>/...
SHARD_ARG = f"--shard {SHARD}" if SHARD else ""

## 3. Unit tests (about a minute)

In [ ]:
!python -m pytest tests/unit -q -p no:cacheprovider 2>&1 | tail -3

## 4. Dry run: records, chunks, estimated size and GPU hours

In [ ]:
!python scripts/legal_data/vectorize.py --input {INPUT} --vectordb {VECTORDB} --categories {CATEGORIES} {SHARD_ARG} --dry-run

## 5. Sample: 50 records per category into a scratch store, then one probe each

In [ ]:
PROBES = {"laws": "מה דינו של חוזה שנכרת בטעות?", "procedural_rules": "מה המועד להגשת כתב הגנה?",
          "supreme_court": "ביטול חוזה בשל טעות"}
if SAMPLE_FIRST:
    !python scripts/legal_data/vectorize.py --input {INPUT} --vectordb /tmp/sample_vectordb --categories {CATEGORIES} --sample 50 --device cuda
    for category in CATEGORIES.split(","):
        question = PROBES[category]
        !python scripts/legal_data/vectorize.py --vectordb /tmp/sample_vectordb --probe "{question}" --category {category}

## 6. Full run (resumable: re-running continues where it stopped)

In [ ]:
!python scripts/legal_data/vectorize.py --input {INPUT} --vectordb {VECTORDB} --categories {CATEGORIES} {SHARD_ARG} --devices {DEVICES}

## 7. Probes with metadata filters

In [ ]:
FILTERS = {"laws": '{"status": "in_force"}', "procedural_rules": '{"authority_level": "regulation"}',
           "supreme_court": '{"decision_ymd": {"$gte": 20150101}}'}
for category in CATEGORIES.split(","):
    question, where = PROBES[category], FILTERS[category]
    !python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --probe "{question}" --category {category} --where '{where}'

## 8. Output: one zip per category

In [ ]:
suffix = f"_shard{SHARD.replace('/', 'of')}" if SHARD else ""
!python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {CATEGORIES} --export-lexical
for category in CATEGORIES.split(","):
    shutil.make_archive(f"/kaggle/working/legal_corpus_{category}{suffix}", "zip", VECTORDB, category)
!ls -lh /kaggle/working/*.zip